# GR00T N1.7 — Step 3: Evaluation

This notebook evaluates the fine-tuned GR00T model locally using open-loop evaluation.

**Prerequisites:**
- Run `01_data_preparation.ipynb` (local dataset at `./datasets/bridge_lerobot`)
- Run `02_training_job.ipynb` and download the model artifacts
- GPU instance with 48GB+ VRAM (g6e.48xlarge or p4d.24xlarge)

**What this does:**
1. Sets up Isaac-GR00T locally for evaluation
2. Evaluates the fine-tuned checkpoint on training data
3. Evaluates the base model for comparison
4. Creates a held-out test set and evaluates generalization
5. Compares results

## 1. Environment Setup

Clone Isaac-GR00T and install dependencies. Only needed once per instance.

In [ ]:
%%bash
set -e

# === System deps ===
sudo apt-get update -qq && sudo apt-get install -y -qq libgl1-mesa-glx libglib2.0-0 git-lfs ffmpeg 2>/dev/null || true

# === Pin torch+torchvision (MUST use PyTorch index to avoid getting torch 2.11+) ===
pip install -q torch==2.7.1 torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cu124
pip install -q optree==0.13.1 --force-reinstall --no-deps

# === Install other deps (do NOT reinstall torch here) ===
pip install -q \
    'transformers>=4.40.0,<4.52' \
    'huggingface-hub>=0.30.0,<1.0' \
    'pyarrow>=14.0' \
    'av' \
    'opencv-python-headless' \
    'datasets' \
    'accelerate' \
    'tyro' \
    'albumentations==1.4.18'

# === Install flash-attn (build from source against local torch) ===
pip install flash-attn==2.7.4.post1 --no-cache-dir --no-build-isolation 2>/dev/null || echo '[WARN] flash-attn build failed'

# === Clone Isaac-GR00T (n1.7-release for GR00T-N1.7-3B) ===
if [ -d "Isaac-GR00T" ]; then
    cd Isaac-GR00T
    CURRENT_TAG=$(git describe --tags --exact-match 2>/dev/null || echo 'none')
    if [ "$CURRENT_TAG" != "n1.7-release" ]; then
        echo '[INFO] Switching to n1.7-release...'
        git fetch --tags
        git checkout n1.7-release
        git submodule update --init --recursive
    fi
else
    git clone --recurse-submodules --branch n1.7-release https://github.com/NVIDIA/Isaac-GR00T.git
    cd Isaac-GR00T
fi
git submodule update --init --recursive

# === Patch and install GR00T ===
PYPROJECT="pyproject.toml"
cp "$PYPROJECT" "${PYPROJECT}.bak" 2>/dev/null || true
sed -i '/tensorrt/d' "$PYPROJECT"
sed -i '/onnx/d' "$PYPROJECT"
sed -i 's/requires-python.*==3\.10.*/requires-python = ">=3.10"/' "$PYPROJECT"
pip install -e . --no-build-isolation 2>&1 | tail -3
mv "${PYPROJECT}.bak" "$PYPROJECT" 2>/dev/null || true

# === Force huggingface-hub<1.0 AFTER GR00T install (GR00T pulls in >=1.0) ===
pip install -q 'huggingface-hub>=0.30.0,<1.0' --force-reinstall --no-deps

# === Verify ===
python3 -c "import gr00t; print('gr00t OK')"
python3 -c "import flash_attn; print(f'flash_attn {flash_attn.__version__} OK')"
python3 -c "import torch; print(f'torch {torch.__version__} OK')"
python3 -c "import torchvision; print(f'torchvision {torchvision.__version__} OK')"
echo '=== Setup complete ==='

In [ ]:
from getpass import getpass
from huggingface_hub import login

# Prompt for Hugging Face token (input is hidden)
hf_token = getpass("Enter your Hugging Face token: ")
login(token=hf_token)

# Download base model weights
from huggingface_hub import snapshot_download
path = snapshot_download("nvidia/GR00T-N1.7-3B")
print(f"Base model at: {path}")

## 2. Configuration

In [ ]:
import os, glob, boto3

# Auto-detect the latest completed GR00T training job
sm_client = boto3.client('sagemaker')
job_resp = sm_client.list_training_jobs(
    NameContains='groot-n17-finetune-bridge',
    SortBy='CreationTime', SortOrder='Descending', MaxResults=10)
completed_jobs = [j for j in job_resp['TrainingJobSummaries'] if j['TrainingJobStatus'] == 'Completed']
if completed_jobs:
    TRAINING_JOB_NAME = completed_jobs[0]['TrainingJobName']
    print(f'Auto-detected training job: {TRAINING_JOB_NAME}')
else:
    TRAINING_JOB_NAME = 'UNKNOWN'
    print('WARNING: No completed training jobs found. Set TRAINING_JOB_NAME manually.')

# Path to the fine-tuned checkpoint
CHECKPOINT_PATH = str(os.path.abspath(f'./model_artifacts/{TRAINING_JOB_NAME}/extracted/bridge_finetune/checkpoint-2000'))

# Fallback: search for any checkpoint dir if exact path doesn't exist
if not os.path.isdir(CHECKPOINT_PATH):
    candidates = glob.glob(f'./model_artifacts/{TRAINING_JOB_NAME}/extracted/**/checkpoint-*', recursive=True)
    if candidates:
        CHECKPOINT_PATH = str(os.path.abspath(candidates[0]))
        print(f'Using checkpoint: {CHECKPOINT_PATH}')
    else:
        print(f'WARNING: No checkpoint found. Contents:')
        extracted = f'./model_artifacts/{TRAINING_JOB_NAME}/extracted/'
        if os.path.isdir(extracted):
            for root, dirs, files in os.walk(extracted):
                for d in dirs[:10]:
                    print(f'  {os.path.join(root, d)}')

# Dataset paths
TRAIN_DATASET = "./datasets/bridge_lerobot"
TEST_DATASET = "./datasets/bridge_lerobot_test"

# Evaluation params
TRAJ_IDS = "0 1 2 3 4"
ACTION_HORIZON = 16

# Modality config
MODALITY_CONFIG = "./scripts/utils/bridge_modality_config.py"

print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Train dataset: {TRAIN_DATASET}")
print(f"Test dataset: {TEST_DATASET}")

## 3. Evaluate Fine-Tuned Model (Training Data)

In [ ]:
import os
TRAIN_DATASET = os.path.abspath("./datasets/bridge_lerobot")
TEST_DATASET = os.path.abspath("./datasets/bridge_lerobot_test")
os.environ["TRAIN_DATASET"] = TRAIN_DATASET
os.environ["TEST_DATASET"] = TEST_DATASET
print(f"TRAIN_DATASET = '{TRAIN_DATASET}'")


In [ ]:
import os
os.environ["CHECKPOINT_PATH"] = CHECKPOINT_PATH
os.environ["TRAIN_DATASET"] = TRAIN_DATASET
os.environ["TEST_DATASET"] = TEST_DATASET
os.environ["TRAJ_IDS"] = TRAJ_IDS
os.environ["ACTION_HORIZON"] = str(ACTION_HORIZON)


In [ ]:
%%bash
cd Isaac-GR00T
echo "=== Evaluating fine-tuned model on training data ==="
python gr00t/eval/open_loop_eval.py \
    --dataset-path "$TRAIN_DATASET" \
    --embodiment-tag NEW_EMBODIMENT \
    --model-path "$CHECKPOINT_PATH" \
    --traj-ids $TRAJ_IDS \
    --action-horizon $ACTION_HORIZON
echo "=== Done ==="

## 4. Evaluate Base Model (for comparison)

The base model doesn't know about our custom embodiment, so we monkey-patch the processor.

In [ ]:
import sys
sys.path.insert(0, "Isaac-GR00T")

# Build bridge modality config
from gr00t.data.types import ModalityConfig, ActionConfig, ActionRepresentation, ActionType, ActionFormat

bridge_modality = {
    "video": ModalityConfig(delta_indices=[0], modality_keys=["front"]),
    "state": ModalityConfig(delta_indices=[0], modality_keys=["arm"]),
    "action": ModalityConfig(
        delta_indices=list(range(0, 16)),
        modality_keys=["arm"],
        action_configs=[ActionConfig(
            rep=ActionRepresentation.ABSOLUTE,
            type=ActionType.NON_EEF,
            format=ActionFormat.DEFAULT,
        )],
    ),
    "language": ModalityConfig(delta_indices=[0], modality_keys=["annotation.human.task_description"]),
}

# Monkey-patch Gr00tPolicy
from gr00t.policy.gr00t_policy import Gr00tPolicy
import torch
import numpy as np
from pathlib import Path
from transformers import AutoModel, AutoProcessor

def _patched_init(self, embodiment_tag, model_path, *, device, strict=True):
    super(Gr00tPolicy, self).__init__(strict=strict)
    model_dir = Path(model_path)
    model = AutoModel.from_pretrained(model_dir, trust_remote_code=True)
    model.eval()
    model.to(device=device, dtype=torch.bfloat16)
    self.model = model
    self.processor = AutoProcessor.from_pretrained(model_dir, trust_remote_code=True)
    self.processor.eval()

    tag = embodiment_tag.value
    configs = self.processor.get_modality_configs()
    if tag not in configs:
        configs[tag] = bridge_modality

    sap = self.processor.state_action_processor
    if hasattr(sap, 'modality_configs') and tag not in sap.modality_configs:
        sap.modality_configs[tag] = bridge_modality

    if hasattr(sap, 'norm_params') and tag not in sap.norm_params:
        arm_stats = {
            'min': np.array([-0.0002622, -0.00387925, 0., 0., 0., 0., 0.]),
            'max': np.array([0.0006804, 0.00021, 0., 0., 0., 0., 0.]),
            'dim': np.array(7),
            'mean': np.array([-6.32212107e-07, 2.15603291e-06, 0., 0., 0., 0., 0.]),
            'std': np.array([2.25099247e-05, 3.73347902e-05, 0., 0., 0., 0., 0.]),
        }
        sap.norm_params[tag] = {
            'state': {'arm': dict(arm_stats)},
            'action': {'arm': dict(arm_stats)},
        }

    self.embodiment_tag = embodiment_tag
    self.modality_configs = configs[embodiment_tag.value]
    self.collate_fn = self.processor.collator
    language_keys = self.modality_configs["language"].modality_keys
    language_delta_indices = self.modality_configs["language"].delta_indices
    assert len(language_keys) == 1
    assert len(language_delta_indices) == 1
    self.language_key = language_keys[0]

Gr00tPolicy.__init__ = _patched_init
print("Gr00tPolicy patched for base model evaluation.")

In [ ]:
# Force GR00T model registration in the current process
import gr00t.model  # This registers Gr00tN1d6 with transformers AutoModel

import runpy, sys
import gr00t.policy.gr00t_policy as gp
gp.Gr00tPolicy.__init__ = _patched_init

traj_ids = TRAJ_IDS.split()
sys.argv = [
    "open_loop_eval.py",
    "--dataset-path", TRAIN_DATASET,
    "--embodiment-tag", "NEW_EMBODIMENT",
    "--model-path", "nvidia/GR00T-N1.7-3B",
    "--traj-ids",
] + traj_ids + [
    "--action-horizon", str(ACTION_HORIZON),
]

print("Running base model evaluation...")
runpy.run_path("Isaac-GR00T/gr00t/eval/open_loop_eval.py", run_name="__main__")
print("Base model evaluation complete.")


## 5. Create Held-Out Test Dataset & Evaluate Generalization

Creates 100 test episodes from BridgeData indices 600-699 (never seen during training).

In [ ]:
# Run the test dataset creation script
# This reuses the same logic from 01_data_preparation but for episodes 600-699
import subprocess
TEST_DATASET = os.path.abspath('./datasets/bridge_lerobot_test')
subprocess.run(f'python scripts/utils/create_groot_test_dataset.py --output-path "{TEST_DATASET}"', shell=True, check=True)
print(f'Test dataset created at: {TEST_DATASET}')
print(f'Exists: {os.path.isdir(TEST_DATASET)}')

In [ ]:
import os
os.environ["CHECKPOINT_PATH"] = CHECKPOINT_PATH
os.environ["TEST_DATASET"] = TEST_DATASET
os.environ["TRAJ_IDS"] = TRAJ_IDS
os.environ["ACTION_HORIZON"] = str(ACTION_HORIZON)


In [ ]:
%%bash -s "$CHECKPOINT_PATH" "$TEST_DATASET" "$TRAJ_IDS" "$ACTION_HORIZON"
cd Isaac-GR00T

echo "=== Evaluating fine-tuned model on TEST data ==="
python gr00t/eval/open_loop_eval.py \
    --dataset-path "$2" \
    --embodiment-tag NEW_EMBODIMENT \
    --model-path "$1" \
    --traj-ids $3 \
    --action-horizon $4

echo "=== Done ==="

In [ ]:
# Base model on test set
sys.argv = [
    "open_loop_eval.py",
    "--dataset-path", TEST_DATASET,
    "--embodiment-tag", "NEW_EMBODIMENT",
    "--model-path", "nvidia/GR00T-N1.7-3B",
    "--traj-ids",
] + traj_ids + [
    "--action-horizon", str(ACTION_HORIZON),
]

print("Running base model evaluation on test set...")
runpy.run_path("Isaac-GR00T/gr00t/eval/open_loop_eval.py", run_name="__main__")
print("Done.")

## 6. Results Summary

Fill in the numbers from the evaluation outputs above.

### Expected Results (from our run: 600 episodes, 2,000 steps, 8× L40S)

| Metric | Base Model | Fine-Tuned | Improvement |
|--------|-----------|------------|-------------|
| MSE (train) | 1.031e-06 | 5.983e-09 | 172× smaller (99.4% ↓) |
| MAE (train) | 3.782e-04 | 3.103e-05 | 12.2× smaller (91.8% ↓) |
| MSE (test)  | 1.066e-06 | 5.318e-09 | 200× smaller (99.5% ↓) |
| MAE (test)  | 3.766e-04 | 3.008e-05 | 12.5× smaller (92.0% ↓) |

Test set performance matches training set → model generalizes, doesn't just memorize.